# Scraping: récupération de la page d'accueil de la RTBF

Dans ce notebook, nous créons un robot qui va ouvrir la page d'accueil du site du journal [La RTBF](https://www.rtbf.be/) et récupérer le titre de tous les articles du jour et les stocker dans un fichier csv.

## Imports

In [1]:
import os
import re
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd


In [3]:
from urllib.parse import urljoin

## Récupération de tous les articles de la page d'accueil



In [4]:

headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}

root_url = "https://max.de.wilde.web.ulb.be/camille/"
response = requests.get(root_url, headers=headers, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')

pdf_urls = []
for link in soup.find_all("a", href=True):
    href = link["href"]
    if href.lower().endswith(".pdf"):
        pdf_urls.append(urljoin(root_url, href))

print(len(pdf_urls))  # doit afficher 51
print(pdf_urls[:3])

51
['https://max.de.wilde.web.ulb.be/camille/KB_JB230_1892-08-07_01-0003.pdf', 'https://max.de.wilde.web.ulb.be/camille/KB_JB427_1920-01-10_01-00004.pdf', 'https://max.de.wilde.web.ulb.be/camille/KB_JB555_1836-02-08_01-00002.pdf']


In [ ]:
# Affichage du nombre de pdf  récupérés
len(pdf_urls) 

51

In [ ]:
# Affichage des 10 premières pdf
pdf_urls[:10]

['https://max.de.wilde.web.ulb.be/camille/KB_JB230_1892-08-07_01-0003.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB427_1920-01-10_01-00004.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB555_1836-02-08_01-00002.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB638_1860-05-21_01-00002.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB773_1918-11-30_01-00002.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB838_1887-12-28_01-00003.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB230_1903-10-16_01-0002.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB427_1933-01-04_01-00003.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB555_1899-01-19_01-00003.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB638_1902-12-20_01-00002.pdf']

## Créez le dossier de sortie puis téléchargez les fichiers pdfs

In [16]:
output_dir = "../../data/camille"  # hors du dépôt
os.makedirs(output_dir, exist_ok=True)

In [17]:
# Compteur de réussites et liste des erreurs
ok, erreurs = 0, []

for i, url in enumerate(pdf_urls, start=1):
    # Nom du fichier local, pris à la fin de l'URL
    filename = os.path.join(output_dir, url.split("/")[-1])

    # Si le fichier est déjà là, on passe au suivant
    if os.path.exists(filename) and os.path.getsize(filename) > 0:
        ok += 1
        continue

    try:
        # Téléchargement du PDF
        r = requests.get(url, headers=headers, timeout=60)
        r.raise_for_status()

        # Vérifie que le fichier est bien un PDF
        if not r.content.startswith(b"%PDF"):
            raise ValueError("le fichier reçu n'est pas un PDF")

        # Enregistrement du fichier
        with open(filename, "wb") as f:
            f.write(r.content)
        ok += 1
        print(f"[{i}/{len(pdf_urls)}] {filename}")

    except Exception as e:
        # En cas d'erreur, on garde l'URL et le message, puis on continue
        erreurs.append((url, str(e)))
        print(f"[{i}/{len(pdf_urls)}] ERREUR {url} : {e}")

    # Pause d'une seconde entre les requêtes
    time.sleep(1)

[1/51] ../../data/camille/KB_JB230_1892-08-07_01-0003.pdf
[2/51] ../../data/camille/KB_JB427_1920-01-10_01-00004.pdf
[3/51] ../../data/camille/KB_JB555_1836-02-08_01-00002.pdf
[4/51] ../../data/camille/KB_JB638_1860-05-21_01-00002.pdf
[5/51] ../../data/camille/KB_JB773_1918-11-30_01-00002.pdf
[6/51] ../../data/camille/KB_JB838_1887-12-28_01-00003.pdf
[7/51] ../../data/camille/KB_JB230_1903-10-16_01-0002.pdf
[8/51] ../../data/camille/KB_JB427_1933-01-04_01-00003.pdf
[9/51] ../../data/camille/KB_JB555_1899-01-19_01-00003.pdf
[10/51] ../../data/camille/KB_JB638_1902-12-20_01-00002.pdf
[11/51] ../../data/camille/KB_JB773_1933-10-07_01-00007.pdf
[12/51] ../../data/camille/KB_JB838_1911-08-03_01-00006.pdf
[13/51] ../../data/camille/KB_JB230_1913-07-05_01-0001.pdf
[14/51] ../../data/camille/KB_JB427_1949-07-18_01-00008.pdf
[15/51] ../../data/camille/KB_JB555_1940-03-01_01-00004.pdf
[16/51] ../../data/camille/KB_JB638_1946-07-18_01-00003.pdf
[17/51] ../../data/camille/KB_JB773_1950-07-22_01-00

## Pour en savoir plus

- Le web scraping avec Python: https://realpython.com/beautiful-soup-web-scraper-python/
- Tutoriel sur les expressions régulières: https://www.w3schools.com/python/python_regex.asp